# Notebook 4 — ECG Heart Rate Variability · `04_ECG_Staging.ipynb`

**Signal:** `ECG` (single lead, from PSG belt electrodes)
**Task:** 5-class AASM sleep staging (Wake, N1, N2, N3, REM)
**Key insight:** Autonomic nervous system patterns are distinct across sleep stages.

---

## Why ECG / HRV?

The autonomic nervous system switches its balance during sleep:

| Stage | ANS Tone | HR | HRV HF | LF/HF | Mechanism |
|---|---|---|---|---|---|
| **Wake** | Mixed | 65–80 bpm | Moderate | ~2.0 | Conscious breathing, activity |
| **N1** | Para starts ↑ | Slight ↓ | Slight ↑ | ↓ | Parasympathetic onset |
| **N2** | Parasympathetic | ↓ | ↑ | Low | Vagal tone, sleep spindles cause HR slowing |
| **N3** | Peak Parasympathetic | Lowest | **Highest** | **Lowest** | Deep vagal dominance, slow deep breathing |
| **REM** | Highly Variable | Variable | **Lowest** | **Highest** | Autonomic storms — sympathetic surges during dream sequences |

**HRV = variability of the interval between consecutive heartbeats (RR intervals)**

High HF power = parasympathetic (N3 marker)
High LF/HF ratio = sympathetic dominance (REM marker)
Low RMSSD = autonomic instability (Wake/REM)

---

## Pipeline Overview
```
EDF → ECG raw → Bandpass filter → R-peak detection (neurokit2)
→ RR intervals per epoch → 10 HRV features
→ List of dicts → DataFrame  ← (explicit checkpoint)
→ EDA → LOSO RF + GRU → save Parquet
```

## Step 1 — Setup: Imports, Seeds, Versions

In [ ]:
import sys, os
PROJECT_ROOT = '/Users/omsrivastava/Documents/healthcare_project'
os.chdir(PROJECT_ROOT)  # ensure CWD = project root for psg_utils
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 110

from scipy.signal import butter, filtfilt, iirnotch, welch
from scipy.interpolate import interp1d
import neurokit2 as nk

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, ConfusionMatrixDisplay
)

import mne
mne.set_log_level('WARNING')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from psg_utils import repro, channel_resolver, epoching, quality, labels, splits, feature_io
repro.fix_seeds()
print('All imports OK.')

## Step 2 — Paths and Constants

In [ ]:
BASE          = os.path.abspath('.')
PSG_DIR       = os.path.join(BASE, 'Sleep_stages', 'PSG')
STAGE_ANN_DIR = os.path.join(BASE, 'Sleep_stages', 'Annotations', 'manual')
RESP_ANN_DIR  = os.path.join(BASE, 'Resp_events',  'Annotations', 'manual')

SUBJECTS   = ['SN1', 'SN2', 'SN3', 'SN4', 'SN5']
FS         = 256
EPOCH_SEC  = 30
EPOCH_SAMP = FS * EPOCH_SEC   # 7680 samples

# ECG bandpass: 0.5–40 Hz removes baseline wander (<0.5 Hz) and
# high-frequency noise (>40 Hz). QRS complex energy is 5–25 Hz.
ECG_BP_LOW  = 0.5
ECG_BP_HIGH = 40.0
NOTCH_FREQ  = 50.0

# HRV frequency bands (Task Force of ESC/NASPE 1996 standard)
VLF_BAND = (0.003, 0.04)   # Very Low Frequency — thermoregulation
LF_BAND  = (0.04,  0.15)   # Low Frequency — sympatho-vagal mix
HF_BAND  = (0.15,  0.40)   # High Frequency — pure vagal (respiratory)

stage_names = {0:'Wake', 1:'N1', 2:'N2', 3:'N3', 4:'REM'}

print('ECG bandpass:', ECG_BP_LOW, '-', ECG_BP_HIGH, 'Hz')
print('HRV bands: VLF', VLF_BAND, '| LF', LF_BAND, '| HF', HF_BAND)

## Step 3 — ECG Signal Filtering

**Bandpass 0.5–40 Hz:** Removes baseline wander (breathing-induced drift, <0.5 Hz)
and high-frequency noise (>40 Hz). All QRS energy lives in 5–25 Hz.

**50 Hz notch:** Removes powerline. Essential in hospital settings.

`filtfilt` = zero-phase (applied forward and backward) → no time shift on QRS peaks.
This matters because R-peak timing directly determines RR intervals.

In [ ]:
def filter_ecg(signal_1d, fs=FS):
    """
    Bandpass 0.5–40 Hz + 50 Hz notch on raw ECG signal.
    Returns filtered signal same shape as input.
    """
    # Bandpass
    b_bp, a_bp = butter(4, [ECG_BP_LOW/(fs/2), ECG_BP_HIGH/(fs/2)], btype='band')
    sig = filtfilt(b_bp, a_bp, signal_1d)

    # 50 Hz notch (Q=30)
    b_n, a_n = iirnotch(NOTCH_FREQ/(fs/2), Q=30)
    sig = filtfilt(b_n, a_n, sig)

    return sig

print('filter_ecg() ready — bandpass 0.5–40 Hz + 50 Hz notch.')

## Step 4 — HRV Feature Extraction (10 features per epoch)

### Pipeline per 30-second epoch:
1. Detect R-peaks using `neurokit2.ecg_peaks()` (Pan-Tompkins algorithm)
2. Compute RR intervals in milliseconds: `RR[i] = (peak[i+1] - peak[i]) / fs * 1000`
3. Time-domain HRV from RR intervals directly
4. Frequency-domain HRV via Welch PSD on linearly interpolated RR signal at 4 Hz
5. Poincaré geometry from successive RR pairs

| Feature | Formula | Stage Signal |
|---|---|---|
| `hr_mean` | 60000 / mean(RR) | N3 lowest, Wake/REM highest |
| `hrv_sdnn` | std(RR) | N3 high, REM very variable |
| `hrv_rmssd` | √mean((RR[i+1]−RR[i])²) | N3 highest (parasympathetic) |
| `hrv_pnn50` | % pairs with \|ΔRRI\| > 50ms | N3 highest |
| `vlf` | PSD integral 0.003–0.04 Hz | Thermoregulation |
| `lf` | PSD integral 0.04–0.15 Hz | Sympatho-vagal mix |
| `hf` | PSD integral 0.15–0.40 Hz | **Pure vagal → N3 peak** |
| `lf_hf_ratio` | LF / HF | **REM peak (autonomic storm)** |
| `poincare_sd1` | std of perpendicular RR scatter | Short-term HRV (vagal) |
| `poincare_sd2` | std of diagonal RR scatter | Long-term HRV (sympatho-vagal) |

In [ ]:
def extract_ecg_features(epoch_1d, fs=FS):
    """
    Extract 10 HRV features from one 30-second ECG epoch.

    Parameters
    ----------
    epoch_1d : np.ndarray  shape (7680,)  filtered ECG
    fs : int

    Returns
    -------
    dict of 10 float features, or NaN-filled dict if R-peaks < 2
    """
    nan_result = {
        'hr_mean': np.nan, 'hrv_sdnn': np.nan, 'hrv_rmssd': np.nan,
        'hrv_pnn50': np.nan, 'vlf': np.nan, 'lf': np.nan,
        'hf': np.nan, 'lf_hf_ratio': np.nan,
        'poincare_sd1': np.nan, 'poincare_sd2': np.nan
    }

    try:
        # ── 1. R-peak detection ───────────────────────────────────────────
        _, info = nk.ecg_peaks(epoch_1d, sampling_rate=fs, method='pantompkins1985')
        rpeaks = info['ECG_R_Peaks']

        if len(rpeaks) < 3:
            return nan_result  # Need at least 3 peaks for 2 intervals

        # ── 2. RR intervals in milliseconds ──────────────────────────────
        rr_ms = np.diff(rpeaks) / fs * 1000.0   # shape: (N_peaks - 1,)

        # Filter physiologically implausible RR intervals
        # Human HR range: 20–200 bpm → RR range: 300–3000 ms
        rr_ms = rr_ms[(rr_ms > 300) & (rr_ms < 3000)]
        if len(rr_ms) < 2:
            return nan_result

        # ── 3. Time-domain HRV ────────────────────────────────────────────
        hr_mean   = 60000.0 / np.mean(rr_ms)
        hrv_sdnn  = np.std(rr_ms)

        rr_diff   = np.diff(rr_ms)
        hrv_rmssd = float(np.sqrt(np.mean(rr_diff**2)))
        hrv_pnn50 = float(np.mean(np.abs(rr_diff) > 50.0))

        # ── 4. Frequency-domain HRV (interpolate RR to 4 Hz grid) ────────
        # Interpolation converts unevenly-sampled RR series to uniform time grid
        # required for FFT/Welch PSD. Standard HRV analysis uses 4 Hz.
        rr_times  = np.cumsum(rr_ms) / 1000.0  # cumulative time in seconds
        rr_times  = rr_times - rr_times[0]     # start from 0

        fs_hrv    = 4.0  # Hz for HRV interpolation (ESC/NASPE standard)
        t_uniform = np.arange(0, rr_times[-1], 1.0/fs_hrv)

        if len(t_uniform) < 8:   # Need enough points for Welch
            vlf = lf = hf = 0.0
        else:
            rr_interp = interp1d(rr_times, rr_ms, kind='linear', fill_value='extrapolate')
            rr_uniform = rr_interp(t_uniform)

            freqs, psd = welch(rr_uniform, fs=fs_hrv, nperseg=min(len(rr_uniform), 128))

            def band_power(freqs, psd, fmin, fmax):
                mask = (freqs >= fmin) & (freqs <= fmax)
                return float(np.trapz(psd[mask], freqs[mask])) if mask.sum() > 0 else 0.0

            vlf = band_power(freqs, psd, *VLF_BAND)
            lf  = band_power(freqs, psd, *LF_BAND)
            hf  = band_power(freqs, psd, *HF_BAND)

        lf_hf_ratio = (lf / hf) if hf > 1e-10 else 0.0

        # ── 5. Poincaré plot SD1 and SD2 ──────────────────────────────────
        # SD1 = std of perpendicular axis = short-term HRV (parasympathetic)
        # SD2 = std of diagonal axis       = long-term HRV  (sympathovagal)
        rr1 = rr_ms[:-1]
        rr2 = rr_ms[1:]
        sd1 = float(np.std((rr2 - rr1) / np.sqrt(2)))
        sd2 = float(np.std((rr2 + rr1) / np.sqrt(2)))

        return {
            'hr_mean':       round(float(hr_mean),   4),
            'hrv_sdnn':      round(float(hrv_sdnn),  4),
            'hrv_rmssd':     round(hrv_rmssd,        4),
            'hrv_pnn50':     round(hrv_pnn50,        4),
            'vlf':           round(vlf,              6),
            'lf':            round(lf,               6),
            'hf':            round(hf,               6),
            'lf_hf_ratio':   round(lf_hf_ratio,      4),
            'poincare_sd1':  round(sd1,              4),
            'poincare_sd2':  round(sd2,              4),
        }

    except Exception:
        return nan_result

print('extract_ecg_features() ready — 10 HRV features per epoch.')

## Step 5 — Multi-Subject Feature Extraction Loop

For each subject SN1–SN5:
1. Load EDF → assert 256 Hz → extract `ECG` channel
2. Bandpass + notch filter
3. Segment into 7,680-sample (30s) epochs
4. Quality gate on raw ECG amplitude
5. Build hard + soft stage labels (12 scorers) and apnea labels
6. Extract 10 HRV features per epoch → **append to `records` list**
7. Parquet saved per subject

> The `records` list is a flat list of dicts — one dict per epoch.
> We convert it to a DataFrame in **Step 6** before any modeling.

In [ ]:
all_records = []       # flat list of dicts — one per epoch, all subjects
quality_reports = []

for subject_id in SUBJECTS:
    sep = '='*60
    print(sep)
    print(f'Processing {subject_id}...')
    print(sep)

    # ── Load EDF ─────────────────────────────────────────────────────────
    edf_path = os.path.join(PSG_DIR, f'{subject_id}_SleepStages.edf')
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

    epoching.assert_fs(raw, subject_id)

    # ── Extract + filter ECG ─────────────────────────────────────────────
    ecg_raw = raw.get_data(picks=['ECG'])[0]      # shape: (L,)
    ecg_filt = filter_ecg(ecg_raw)

    # ── Epoch segmentation ───────────────────────────────────────────────
    ecg_epochs = epoching.make_epochs(ecg_filt, subject_id)
    n_epochs = len(ecg_epochs)

    # ── Quality gate ─────────────────────────────────────────────────────
    quality_mask, qreport = quality.gate_epochs(ecg_epochs, subject_id, 'ECG')
    quality_reports.append(qreport)

    # ── Build labels ─────────────────────────────────────────────────────
    hard_labels, soft_labels, tie_count, n_scorers = labels.build_stage_labels(
        subject_id, STAGE_ANN_DIR, n_epochs
    )
    apnea_labels, n_events, ahi = labels.build_apnea_labels(
        subject_id, RESP_ANN_DIR, n_epochs
    )

    # ── Extract features → append to records list ─────────────────────────
    n_nan = 0
    for i in range(n_epochs):
        feat = extract_ecg_features(ecg_epochs[i])

        if np.isnan(feat['hr_mean']):
            n_nan += 1

        record = {
            'subject':     subject_id,
            'epoch_idx':   i,
            'hard_label':  int(hard_labels[i]),
            'apnea_label': int(apnea_labels[i]),
            'soft_W':      float(soft_labels[i, 0]),
            'soft_N1':     float(soft_labels[i, 1]),
            'soft_N2':     float(soft_labels[i, 2]),
            'soft_N3':     float(soft_labels[i, 3]),
            'soft_REM':    float(soft_labels[i, 4]),
            'quality_ok':  bool(quality_mask[i]),
        }
        record.update(feat)
        all_records.append(record)

    print(f'{subject_id}: {n_epochs} epochs | {n_scorers} scorers | '
          f'{int(quality_mask.sum())} quality-OK | {n_nan} NaN-HRV | AHI={ahi:.1f}')

print()
print(f'Total records collected: {len(all_records)}')
print('Next: convert to DataFrame.')

## Step 6 — ✅ Convert to DataFrame (Checkpoint)

**All raw extraction is done. From here we work entirely with `ecg_df`.**

This is the single source of truth for all downstream steps:
- EDA visualizations
- LOSO splits
- RF and GRU training
- `feature_io.save_features()` export

No more array indexing, no more per-subject loops — just `ecg_df`.

In [ ]:
# ── Convert list of dicts → DataFrame ────────────────────────────────────
ecg_df = pd.DataFrame(all_records)

print('Shape:', ecg_df.shape)
print('Columns:', list(ecg_df.columns))
print()

# ── Preview first rows ───────────────────────────────────────────────────
print('=== HEAD (first 5 rows) ===')
print(ecg_df.head().to_string())
print()

# ── Data types and memory ────────────────────────────────────────────────
print('=== INFO ===')
ecg_df.info()
print()

# ── Statistical summary of the 10 HRV features ───────────────────────────
ECG_FEATURES = ['hr_mean','hrv_sdnn','hrv_rmssd','hrv_pnn50',
                'vlf','lf','hf','lf_hf_ratio','poincare_sd1','poincare_sd2']

print('=== DESCRIBE (10 HRV features) ===')
print(ecg_df[ECG_FEATURES].describe().round(4).to_string())
print()

# ── NaN audit ────────────────────────────────────────────────────────────
nan_counts = ecg_df[ECG_FEATURES].isna().sum()
print('=== NaN counts per feature ===')
print(nan_counts.to_string())
print()
print(f'Total NaN epochs: {ecg_df[ECG_FEATURES[0]].isna().sum()} / {len(ecg_df)}')

## Step 7 — Clean DataFrame and Save per Subject to Parquet

Two filters applied:
1. **quality_ok == True** — removes signal quality failures
2. **Drop NaN HRV rows** — epochs where R-peak detection found fewer than 3 peaks
   (happens in very short-duration arrhythmia bursts or artifact epochs)

We save Parquet **before** modeling so future notebooks (esp. Fusion) can load
pre-computed features without re-running the expensive filter+peak detection step.

In [ ]:
# ── Apply quality gate ────────────────────────────────────────────────────
ecg_clean = ecg_df[ecg_df['quality_ok']].copy()

# ── Drop NaN HRV epochs ───────────────────────────────────────────────────
n_before = len(ecg_clean)
ecg_clean = ecg_clean.dropna(subset=ECG_FEATURES).reset_index(drop=True)
n_after = len(ecg_clean)

print(f'After quality gate: {n_before} epochs')
print(f'After NaN drop:     {n_after} epochs (removed {n_before - n_after} NaN-HRV epochs)')
print(f'Stage distribution:')
print(ecg_clean['hard_label'].map(stage_names).value_counts().sort_index())
print()

# ── Save per-subject Parquet ──────────────────────────────────────────────
for subject_id in SUBJECTS:
    df_subj = ecg_clean[ecg_clean['subject'] == subject_id].copy()
    feature_io.save_features(df_subj, signal_name='ecg', subject_id=subject_id)

print('All subject Parquet files saved to features/ecg/')

## Step 8 — Quality Gate Report

In [ ]:
qdf = pd.DataFrame(quality_reports)
print(qdf[['subject','n_total','n_flagged','drop_rate','flat','saturated','outlier']].to_string(index=False))

## Step 9 — EDA: HRV Features by Sleep Stage

Expected patterns per feature:
- `hr_mean`: Lowest in N3, highest in Wake/REM
- `hf`: Highest in N3 (peak parasympathetic)
- `lf_hf_ratio`: Highest in REM (autonomic storm) and Wake
- `hrv_rmssd` + `poincare_sd1`: Highest in N3

If these patterns don't appear, it may indicate ECG lead-off artifacts or
that our bandpass/R-peak pipeline needs adjustment.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('ECG/HRV Features by Sleep Stage (all 5 subjects combined)',
             fontsize=13, fontweight='bold')

stage_labels_ordered = ['Wake', 'N1', 'N2', 'N3', 'REM']
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6']

for ax, feat in zip(axes.flatten(), ECG_FEATURES):
    data = [ecg_clean[ecg_clean['hard_label'] == s][feat].dropna().values
            for s in range(5)]
    bp = ax.boxplot(data, tick_labels=stage_labels_ordered,
                    patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('Stage')
    ax.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig('ecg_eda_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA saved: ecg_eda_boxplots.png')

## Step 10 — LOSO Random Forest Baseline

Working directly from `ecg_clean` DataFrame — no raw arrays, no index gymnastics.
`splits.loso_splits(ecg_clean)` splits on the `subject` column automatically.

Reporting metrics against **both** hard-vote and soft-argmax labels per the master prompt.

In [ ]:
rf_results = []

for train_df, test_df, test_subject in splits.loso_splits(ecg_clean):

    X_train, y_train = splits.get_X_y(train_df, ECG_FEATURES, 'hard_label')
    X_test,  y_test  = splits.get_X_y(test_df,  ECG_FEATURES, 'hard_label')
    y_soft = test_df[['soft_W','soft_N1','soft_N2','soft_N3','soft_REM']].values.argmax(axis=1)

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=20, max_features='log2',
        criterion='entropy', class_weight='balanced',
        n_jobs=-1, random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred, average='macro', zero_division=0)
    kappa = cohen_kappa_score(y_test, y_pred)
    pcf   = f1_score(y_test, y_pred, average=None, zero_division=0, labels=[0,1,2,3,4])

    acc_s = accuracy_score(y_soft, y_pred)
    f1_s  = f1_score(y_soft, y_pred, average='macro', zero_division=0)
    k_s   = cohen_kappa_score(y_soft, y_pred)

    rf_results.append({'subject': test_subject,
        'acc_hard': acc, 'f1_hard': f1, 'kappa_hard': kappa,
        'acc_soft': acc_s, 'f1_soft': f1_s, 'kappa_soft': k_s,
        'f1_Wake': pcf[0], 'f1_N1': pcf[1], 'f1_N2': pcf[2],
        'f1_N3': pcf[3], 'f1_REM': pcf[4], 'model': 'RF'})

    print(f'{test_subject}: acc={acc:.4f} F1={f1:.4f} k={kappa:.4f} | '
          f'soft F1={f1_s:.4f} k={k_s:.4f}')
    print(f'  Per-class: W={pcf[0]:.2f} N1={pcf[1]:.2f} N2={pcf[2]:.2f} N3={pcf[3]:.2f} REM={pcf[4]:.2f}')

rf_df = pd.DataFrame(rf_results)
print()
print('=== RF LOSO MEAN (Hard) ===')
print(f'Accuracy: {rf_df.acc_hard.mean():.4f} +/- {rf_df.acc_hard.std():.4f}')
print(f'Macro F1: {rf_df.f1_hard.mean():.4f} +/- {rf_df.f1_hard.std():.4f}')
print(f'Cohen k:  {rf_df.kappa_hard.mean():.4f} +/- {rf_df.kappa_hard.std():.4f}')
print(f'N3 F1:    {rf_df.f1_N3.mean():.4f}   REM F1: {rf_df.f1_REM.mean():.4f}')

## Step 11 — Confusion Matrix (Best RF Fold)

In [ ]:
best_subj = rf_df.loc[rf_df['f1_hard'].idxmax(), 'subject']
print(f'Best RF fold: {best_subj}')

train_b = ecg_clean[ecg_clean['subject'] != best_subj]
test_b  = ecg_clean[ecg_clean['subject'] == best_subj]
Xtr, ytr = splits.get_X_y(train_b, ECG_FEATURES, 'hard_label')
Xte, yte = splits.get_X_y(test_b,  ECG_FEATURES, 'hard_label')

rf_b = RandomForestClassifier(n_estimators=200, max_depth=20, max_features='log2',
    criterion='entropy', class_weight='balanced', n_jobs=-1, random_state=42)
rf_b.fit(Xtr, ytr)
yp = rf_b.predict(Xte)

print(classification_report(yte, yp, target_names=['Wake','N1','N2','N3','REM'], zero_division=0))

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(
    yte, yp, display_labels=['Wake','N1','N2','N3','REM'],
    cmap='Blues', ax=ax)
ax.set_title(f'RF Confusion Matrix — Test: {best_subj}', fontweight='bold')
plt.tight_layout()
plt.savefig('ecg_rf_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 12 — Feature Importance

Expected: `hf` (HF power) and `hrv_rmssd` should rank high — they directly encode
the N3 parasympathetic signature. `lf_hf_ratio` should also rank high as the REM marker.

In [ ]:
imp_df = pd.DataFrame({
    'feature': ECG_FEATURES,
    'importance': rf_b.feature_importances_
}).sort_values('importance', ascending=False)

print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(imp_df['feature'], imp_df['importance'], color='#2c82c9', alpha=0.85)
ax.set_title('RF Feature Importance — ECG/HRV', fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('ecg_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 13 — LOSO GRU Sequential Model (seq_len=5)

Same architecture as Notebook 3. Input is the `ecg_clean` DataFrame — we just
call `splits.loso_splits()` and `splits.make_sequences()` on it.

ECG HRV features are already computed per epoch; the GRU adds temporal context
— e.g., **steadily increasing HF power** signals the descent into N3.

In [ ]:
SEQ_LEN = 5
gru_results = []

for train_df, test_df, test_subject in splits.loso_splits(ecg_clean):

    X_tr_tab, y_tr = splits.get_X_y(train_df, ECG_FEATURES, 'hard_label')
    X_te_tab, y_te = splits.get_X_y(test_df,  ECG_FEATURES, 'hard_label')
    y_soft = test_df[['soft_W','soft_N1','soft_N2','soft_N3','soft_REM']].values.argmax(axis=1)

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr_tab)
    X_te_sc = scaler.transform(X_te_tab)

    X_tr_seq, y_tr_seq = splits.make_sequences(X_tr_sc, y_tr, SEQ_LEN)
    X_te_seq, y_te_seq = splits.make_sequences(X_te_sc, y_te, SEQ_LEN)
    y_soft_seq = y_soft[SEQ_LEN-1:]

    tf.random.set_seed(42)
    model = Sequential([
        GRU(64, input_shape=(SEQ_LEN, len(ECG_FEATURES))),
        Dropout(0.3),
        Dense(5, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    es = EarlyStopping(monitor='val_loss', patience=5,
                       restore_best_weights=True, verbose=0)
    model.fit(X_tr_seq, y_tr_seq, epochs=30, batch_size=64,
              validation_split=0.15, callbacks=[es],
              verbose=0, shuffle=False)

    y_pred = model.predict(X_te_seq, verbose=0).argmax(axis=1)

    acc   = accuracy_score(y_te_seq, y_pred)
    f1    = f1_score(y_te_seq, y_pred, average='macro', zero_division=0)
    kappa = cohen_kappa_score(y_te_seq, y_pred)
    pcf   = f1_score(y_te_seq, y_pred, average=None, zero_division=0, labels=[0,1,2,3,4])
    acc_s = accuracy_score(y_soft_seq, y_pred)
    f1_s  = f1_score(y_soft_seq, y_pred, average='macro', zero_division=0)
    k_s   = cohen_kappa_score(y_soft_seq, y_pred)

    gru_results.append({'subject': test_subject,
        'acc_hard': acc, 'f1_hard': f1, 'kappa_hard': kappa,
        'acc_soft': acc_s, 'f1_soft': f1_s, 'kappa_soft': k_s,
        'f1_Wake': pcf[0], 'f1_N1': pcf[1], 'f1_N2': pcf[2],
        'f1_N3': pcf[3], 'f1_REM': pcf[4], 'model': 'GRU_seq5'})

    print(f'{test_subject}: acc={acc:.4f} F1={f1:.4f} k={kappa:.4f} | '
          f'soft F1={f1_s:.4f}')
    print(f'  Per-class: W={pcf[0]:.2f} N1={pcf[1]:.2f} N2={pcf[2]:.2f} N3={pcf[3]:.2f} REM={pcf[4]:.2f}')

gru_df = pd.DataFrame(gru_results)
print()
print('=== GRU LOSO MEAN (Hard) ===')
print(f'Accuracy: {gru_df.acc_hard.mean():.4f} +/- {gru_df.acc_hard.std():.4f}')
print(f'Macro F1: {gru_df.f1_hard.mean():.4f} +/- {gru_df.f1_hard.std():.4f}')
print(f'Cohen k:  {gru_df.kappa_hard.mean():.4f} +/- {gru_df.kappa_hard.std():.4f}')
print(f'N3 F1:    {gru_df.f1_N3.mean():.4f}   REM F1: {gru_df.f1_REM.mean():.4f}')

## Step 14 — Summary Comparison

In [ ]:
print('='*65)
print('ECG/HRV SIGNAL - LOSO BENCHMARK SUMMARY')
print('='*65)
header = '{:<18} {:>10} {:>10} {:>10} {:>8} {:>8}'.format(
    'Model','Acc','Macro F1','Cohen k','N3 F1','REM F1')
print(header)
print('-'*65)
rf_row = '{:<18} {:>10.4f} {:>10.4f} {:>10.4f} {:>8.4f} {:>8.4f}'.format(
    'RF (10 feat)',
    rf_df.acc_hard.mean(), rf_df.f1_hard.mean(), rf_df.kappa_hard.mean(),
    rf_df.f1_N3.mean(), rf_df.f1_REM.mean())
gru_row = '{:<18} {:>10.4f} {:>10.4f} {:>10.4f} {:>8.4f} {:>8.4f}'.format(
    'GRU seq=5',
    gru_df.acc_hard.mean(), gru_df.f1_hard.mean(), gru_df.kappa_hard.mean(),
    gru_df.f1_N3.mean(), gru_df.f1_REM.mean())
print(rf_row)
print(gru_row)
print('-'*65)
print('Human baseline: k ~= 0.76')
print('ECG N3 target:  F1 > EOG N3 (ECG should add N3 info via HF-HRV)')
print()
# Compare to previous signals
print('Signal comparison (Macro F1, best model each):')
print('  EOG:  57.54% (GRU) <- best single signal so far')
print('  EMG:  23.48% (RF)  <- atonia fails LOSO')
ecg_best = max(rf_df.f1_hard.mean(), gru_df.f1_hard.mean())
print(f'  ECG:  {ecg_best:.2%} ({"RF" if rf_df.f1_hard.mean() > gru_df.f1_hard.mean() else "GRU"})')

## Summary — ECG/HRV Signal Clinical Assessment

**Where ECG/HRV excels:**
- **N3 detection** — High-frequency HRV power (HF) is the direct physiological marker
  of peak parasympathetic tone during slow-wave sleep. If `hf` ranks high in feature
  importance, ECG adds unique N3 information not available from EOG or EMG.
- **REM detection** — LF/HF ratio (autonomic imbalance) and low RMSSD signal
  the sympathetic surges of dreaming. Independent of eye movement (EOG) information.

**Where ECG is weak:**
- **N1 vs N2 separation** — Both are parasympathetic with only gradual HRV shifts.
  This boundary is subtle even for cardiologists.
- **Arrhythmia confounds** — Subjects with irregular HR (AFib, frequent PVCs) will
  have elevated SDNN/RMSSD in all stages, breaking the N3 signature.

**Unique contribution to Fusion (Notebook 06):**
- `hf` + `hrv_rmssd` for **N3 specificity** (complementary to EEG delta power)
- `lf_hf_ratio` for **REM disambiguation** (complementary to EOG saccades + EMG atonia)
- `hr_mean` for **apnea correlation** — HR acceleration follows respiratory events

**Next:** Notebook 05 — Respiration + SaO2. This is the only signal notebook
that evaluates **both** staging AND apnea detection simultaneously.